### Imports

In [24]:
import os
import re
import math
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter, defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Primary device : {device}')
if torch.cuda.is_available():
    print(f'GPU count      : {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}        : {torch.cuda.get_device_name(i)}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Primary device : cpu


### Loading and Inspecting the dataset

In [25]:
DATA_PATH = '/kaggle/input/datasets/abdullah23f3061/customer-support-tickets-csv/customer_support_tickets.csv'
MY_PATH = 'dataset/customer_support_tickets.csv'

df = pd.read_csv(MY_PATH)
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [26]:
df.hist

<bound method hist_frame of       Ticket ID        Customer Name              Customer Email  \
0             1        Marisa Obrien  carrollallison@example.com   
1             2         Jessica Rios    clarkeashley@example.com   
2             3  Christopher Robbins   gonzalestracy@example.com   
3             4     Christina Dillon    bradleyolson@example.org   
4             5    Alexander Carroll     bradleymark@example.com   
...         ...                  ...                         ...   
8464       8465           David Todd          adam28@example.net   
8465       8466           Lori Davis       russell68@example.com   
8466       8467      Michelle Kelley        ashley83@example.org   
8467       8468     Steven Rodriguez         fpowell@example.org   
8468       8469      Steven Davis MD          lori20@example.net   

      Customer Age Customer Gender       Product Purchased Date of Purchase  \
0               32           Other              GoPro Hero       2021-03-22 

In [27]:
df.isna().sum()

Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

### Doing Data cleaning

In [28]:
FOCUS = ['Ticket Description', 'Ticket Subject', 'Ticket Priority', 'Ticket Type', 'Ticket Channel']

df = df[FOCUS].copy()
df.dropna(subset=['Ticket Description', 'Ticket Priority', 'Ticket Channel'], inplace=True)
df.reset_index(drop=True, inplace=True)


In [29]:
df.head()

,Ticket Description,Ticket Subject,Ticket Priority,Ticket Type,Ticket Channel
0,I'm having an issue with the {product_purchase...,Product setup,Critical,Technical issue,Social media
1,I'm having an issue with the {product_purchase...,Peripheral compatibility,Critical,Technical issue,Chat
2,I'm facing a problem with my {product_purchase...,Network problem,Low,Technical issue,Social media
3,I'm having an issue with the {product_purchase...,Account access,Low,Billing inquiry,Social media
4,I'm having an issue with the {product_purchase...,Data loss,Low,Billing inquiry,Email


In [30]:

print(df.shape)
print('\npriority value counts:')
print(df['Ticket Priority'].value_counts())
print('\nchannel value counts:')
print(df['Ticket Channel'].value_counts())

(8469, 5)

priority value counts:
Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

channel value counts:
Ticket Channel
Email           2143
Phone           2132
Social media    2121
Chat            2073
Name: count, dtype: int64


### Label Encoding

In [31]:
class LabelEncoder:
    
    def __init__(self, unknown_index: int = -1):
        self.unknown_index = unknown_index
        self.label2idx: dict = {}
        self.idx2label: dict = {}

    def fit(self, labels):
        unique = sorted(set(labels))
        self.label2idx = {lbl: i for i, lbl in enumerate(unique)}
        self.idx2label = {i: lbl for lbl, i in self.label2idx.items()}
        return self

    def transform(self, labels) -> np.ndarray:
        return np.array(
            [self.label2idx.get(lbl, self.unknown_index) for lbl in labels],
            dtype=np.int64
        )

    def fit_transform(self, labels) -> np.ndarray:
        return self.fit(labels).transform(labels)

    def inverse_transform(self, indices) -> list:
        return [self.idx2label.get(i, '<UNKNOWN>') for i in indices]


In [32]:
priority_encoder = LabelEncoder(unknown_index=-1)
priority_encoded  = priority_encoder.fit_transform(df['Ticket Priority'].tolist())

print('label -> index mapping:', priority_encoder.label2idx)
print('first 10 encodings:', priority_encoded[:10])

assert list(priority_encoder.inverse_transform(priority_encoded[:5])) == \
       df['Ticket Priority'].tolist()[:5], 'round-trip check failed!'
print('round-trip check: pass')

unseen = priority_encoder.transform(['Extreme', 'Low'])
print('unseen category test :', unseen)

label -> index mapping: {'Critical': 0, 'High': 1, 'Low': 2, 'Medium': 3}
first 10 encodings: [0 0 2 2 2 2 0 0 2 0]
round-trip check: pass
unseen category test : [-1  2]


### One hot encoding

In [33]:
class OneHotEncoder:
    def __init__(self):
        self.category2idx: dict = {}
        self.num_classes: int   = 0

    def fit(self, categories):
        unique = sorted(set(categories))
        self.category2idx = {c: i for i, c in enumerate(unique)}
        self.num_classes   = len(unique)
        return self

    def transform(self, categories) -> np.ndarray:
        n = len(categories)
        matrix = np.zeros((n, self.num_classes), dtype=np.float32)
        for row, cat in enumerate(categories):
            idx = self.category2idx.get(cat, None)
            if idx is not None:
                matrix[row, idx] = 1.0
        return matrix

    def fit_transform(self, categories) -> np.ndarray:
        return self.fit(categories).transform(categories)

In [34]:
channel_encoder = OneHotEncoder()
channel_onehot  = channel_encoder.fit_transform(df['Ticket Channel'].tolist())

print('category -> index mapping:', channel_encoder.category2idx)
print(f'One-hot matrix shape    : {channel_onehot.shape}')
print('first 5 rows:')
print(channel_onehot[:5])


unseen_oh = channel_encoder.transform(['Telegram', 'Email'])
print('\nunseen category -> all-zeros:', (unseen_oh[0] == 0).all())

category -> index mapping: {'Chat': 0, 'Email': 1, 'Phone': 2, 'Social media': 3}
One-hot matrix shape    : (8469, 4)
first 5 rows:
[[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]

unseen category -> all-zeros: True


### Sparse Representation or TF-IDF

### Tokenization

In [36]:
def tokenize(text: str) -> list:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    return tokens

In [37]:
sample = "I'm having an issue with the GoPro Hero. Please assist!"
print('Sample tokens:', tokenize(sample))

Sample tokens: ['i', 'm', 'having', 'an', 'issue', 'with', 'the', 'gopro', 'hero', 'please', 'assist']


### N GRams

In [39]:
def generate_ngrams(tokens: list, n: int) -> list:
    return ['_'.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

In [40]:
    
def tokenize_with_ngrams(text: str, max_n: int = 3) -> list:
    tokens = tokenize(text)
    combined = list(tokens)
    for n in range(2, max_n + 1):
        combined.extend(generate_ngrams(tokens, n))
    return combined

In [41]:
demo_tokens = tokenize('not working is working fine')
print('Bigrams :', generate_ngrams(demo_tokens, 2))
print('Trigrams:', generate_ngrams(demo_tokens, 3))

Bigrams : ['not_working', 'working_is', 'is_working', 'working_fine']
Trigrams: ['not_working_is', 'working_is_working', 'is_working_fine']


### BOW top 5000 vocabulary

In [42]:
class CountVectorizer:
    def __init__(self, max_features: int = 5000, max_n: int = 3):
        self.max_features = max_features
        self.max_n        = max_n
        self.vocab: dict  = {}       # token → col index
        self.vocab_size: int = 0

    def fit(self, corpus: list):
        counter = Counter()
        for doc in corpus:
            tokens = tokenize_with_ngrams(doc, self.max_n)
            counter.update(tokens)
        top_tokens = [tok for tok, _ in counter.most_common(self.max_features)]
        self.vocab      = {tok: i for i, tok in enumerate(top_tokens)}
        self.vocab_size = len(self.vocab)
        return self

    def transform(self, corpus: list) -> torch.Tensor:
        rows, cols, vals = [], [], []
        for row_idx, doc in enumerate(corpus):
            tokens  = tokenize_with_ngrams(doc, self.max_n)
            cnt     = Counter(tok for tok in tokens if tok in self.vocab)
            for tok, c in cnt.items():
                rows.append(row_idx)
                cols.append(self.vocab[tok])
                vals.append(float(c))

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values  = torch.tensor(vals, dtype=torch.float32)
        return torch.sparse_coo_tensor(
            indices, values,
            size=(len(corpus), self.vocab_size)
        ).coalesce()

    def fit_transform(self, corpus: list) -> torch.Tensor:
        return self.fit(corpus).transform(corpus)

In [43]:
descriptions = df['Ticket Description'].tolist()

cv = CountVectorizer(max_features=5000, max_n=3)
bow_sparse = cv.fit_transform(descriptions)

print(f'Vocabulary size   : {cv.vocab_size}')
print(f'BoW sparse shape  : {bow_sparse.shape}')
print(f'Non-zero elements : {bow_sparse._nnz()}')

Vocabulary size   : 5000
BoW sparse shape  : torch.Size([8469, 5000])
Non-zero elements : 928769


### TF-IDF

In [44]:
class TFIDFVectorizer:
    def __init__(self, max_features: int = 5000, max_n: int = 3):
        self.cv   = CountVectorizer(max_features=max_features, max_n=max_n)
        self.idf_: torch.Tensor = None

    def _compute_idf(self, bow_sparse: torch.Tensor, N: int) -> torch.Tensor:
        bow_dense = bow_sparse.to_dense() # (N, vocab)
        df = (bow_dense > 0).sum(dim=0).float() #(vocab,)
        idf = torch.log((1 + N) / (1 + df)) + 1 #smooth
        return idf

    def fit(self, corpus: list):
        bow_sparse = self.cv.fit_transform(corpus)
        N          = len(corpus)
        self.idf_  = self._compute_idf(bow_sparse, N) #(vocab_size,)
        return self
        
    def transform(self, corpus: list) -> torch.Tensor:
        rows, cols, vals = [], [], []
        for row_idx, doc in enumerate(corpus):
            tokens   = tokenize_with_ngrams(doc, self.cv.max_n)
            doc_len  = max(len(tokens), 1)
            cnt      = Counter(tok for tok in tokens if tok in self.cv.vocab)
            for tok, c in cnt.items():
                col_idx = self.cv.vocab[tok]
                tf      = c / doc_len
                tfidf   = tf * self.idf_[col_idx].item()
                rows.append(row_idx)
                cols.append(col_idx)
                vals.append(tfidf)

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values  = torch.tensor(vals, dtype=torch.float32)
        sparse  = torch.sparse_coo_tensor(
            indices, values,
            size=(len(corpus), self.cv.vocab_size)
        ).coalesce()
        return sparse

    def fit_transform(self, corpus: list) -> torch.Tensor:
        return self.fit(corpus).transform(corpus)

    def transform_single(self, text: str) -> torch.Tensor:
        tokens   = tokenize_with_ngrams(text, self.cv.max_n)
        doc_len  = max(len(tokens), 1)
        cnt      = Counter(tok for tok in tokens if tok in self.cv.vocab)
        vec      = torch.zeros(self.cv.vocab_size, dtype=torch.float32)
        for tok, c in cnt.items():
            col_idx    = self.cv.vocab[tok]
            tf         = c / doc_len
            vec[col_idx] = tf * self.idf_[col_idx].item()
        return vec

In [46]:

print('fitting TF-IDF')
tfidf_vectorizer = TFIDFVectorizer(max_features=5000, max_n=3)
tfidf_sparse     = tfidf_vectorizer.fit_transform(descriptions)

print(f'TF-IDF sparse shape  : {tfidf_sparse.shape}')
print(f'non-zero elements    : {tfidf_sparse._nnz()}')
print(f'IDF vector shape     : {tfidf_vectorizer.idf_.shape}')
print(f'IDF min / max        : {tfidf_vectorizer.idf_.min():.4f} / {tfidf_vectorizer.idf_.max():.4f}')

fitting TF-IDF
TF-IDF sparse shape  : torch.Size([8469, 5000])
non-zero elements    : 928769
IDF vector shape     : torch.Size([5000])
IDF min / max        : 1.0000 / 8.9457


### GloVe download

In [49]:
GLOVE_KAGGLE_DIR = '/kaggle/working/glove'
GLOVE_DIR  = './glove'
GLOVE_URL  = 'https://nlp.stanford.edu/data/glove.6B.zip'
GLOVE_FILE = os.path.join(GLOVE_DIR, 'glove.6B.300d.txt')
GLOVE_DIM  = 300

os.makedirs(GLOVE_DIR, exist_ok=True)

if not os.path.exists(GLOVE_FILE):
    print('downloading GloVe 6B')
    zip_path = os.path.join(GLOVE_DIR, 'glove.6B.zip')
    
    urllib.request.urlretrieve(GLOVE_URL, zip_path)
    
    print('Extracting ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extract('glove.6B.300d.txt', GLOVE_DIR)
    
    os.remove(zip_path)
    print('doneing.............. Yahahahaha RRRRaaaaaaaaaa')
else:
    print('gloVe file already present.')

downloading GloVe 6B
Extracting ...
doneing.............. Yahahahaha RRRRaaaaaaaaaa
